# Guide: Parametric Curves (NS/Svensson)

This notebook runs deterministic curve fits, then compares baseline vs macro-shock states and interprets the implications for P&L, DV01, convexity, basis context, and hedge action.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))

from src.curves.parametric import fit_parametric_curve


In [ ]:
np.random.seed(42)
tenors = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
base_true = 0.06 - 0.03*np.exp(-tenors/2.0) + 0.004*np.sin(tenors)
noise = np.array([0.0, 0.0003, -0.0002, 0.0001, 0.0002, -0.0001, 0.0002, -0.0002])
y_base = base_true + noise

y_shock = y_base + np.array([0.015, 0.012, 0.010, 0.008, 0.007, 0.006, 0.006, 0.005])

fit_base = fit_parametric_curve(tenors, y_base, model='svensson', weight_mode='uniform')
fit_shock = fit_parametric_curve(tenors, y_shock, model='svensson', weight_mode='uniform')

curve_compare = pd.DataFrame({
    'tenor': tenors,
    'base_input': y_base,
    'shock_input': y_shock,
    'base_fit': fit_base.curve(tenors),
    'shock_fit': fit_shock.curve(tenors),
})
curve_compare['delta_shock_minus_base_bp'] = (curve_compare['shock_fit'] - curve_compare['base_fit']) * 1e4
curve_compare


In [ ]:
# Risk interpretation metrics for a stylized 5Y fixed-receiver (long duration) position.
notional = 100_000_000
mod_duration_5y = 4.6
rate_shift = float(curve_compare.loc[curve_compare['tenor']==5.0,'shock_fit'].item() - curve_compare.loc[curve_compare['tenor']==5.0,'base_fit'].item())

pnl_approx = -notional * mod_duration_5y * rate_shift
dv01 = notional * mod_duration_5y * 1e-4

# Convexity proxy: second-order term with an assumed convexity coefficient.
convexity_coeff = 0.32
convexity_pnl = 0.5 * notional * convexity_coeff * (rate_shift**2)

risk_table = pd.DataFrame({
    'metric': ['5Y rate shift', 'DV01', '1st-order P&L', 'Convexity offset'],
    'value': [rate_shift, dv01, pnl_approx, convexity_pnl]
})
risk_table


In [ ]:
class ParametricCurveExplainer:
    def __init__(self, curve_compare: pd.DataFrame, risk_table: pd.DataFrame):
        self.curve_compare = curve_compare
        self.risk_table = risk_table

    def render_full_markdown(self) -> str:
        shock_5y_bp = float(self.curve_compare.loc[self.curve_compare['tenor']==5.0,'delta_shock_minus_base_bp'].item())
        dv01 = float(self.risk_table.loc[self.risk_table['metric']=='DV01','value'].item())
        pnl = float(self.risk_table.loc[self.risk_table['metric']=='1st-order P&L','value'].item())
        conv = float(self.risk_table.loc[self.risk_table['metric']=='Convexity offset','value'].item())
        return f"""
### Interpretation (P&L / DV01 / Convexity / Basis / Hedge Action)
- **Macro shock change:** curve shifts up; 5Y move is about **{shock_5y_bp:.1f} bp**.
- **DV01:** estimated around **{dv01:,.0f}** currency units per 1 bp.
- **P&L:** first-order impact for a long-duration receiver is **{pnl:,.0f}** (negative under up-shock).
- **Convexity:** second-order term partly offsets linear loss by about **{conv:,.0f}**.
- **Basis angle:** if cross-currency basis widens simultaneously, offshore funding hedges can underperform pure rates hedges.
- **Hedge action:** reduce net DV01 using pay-fixed swaps/futures and add basis hedges when funding stress accompanies rate shocks.
""".strip()

explainer = ParametricCurveExplainer(curve_compare, risk_table)
print(explainer.render_full_markdown())
